# Практическая работа №1. Введение в инструментарий анализа данных

## 0. Назначение практической работы

Цель работы - освоить полный базовый цикл анализа табличных данных: подготовить воспроизводимое Python-окружение, загрузить данные, проверить их структуру и качество, выполнить предобработку, получить описательные статистики, сформировать простые аналитические срезы, визуализировать закономерности и сохранить подготовленный набор данных в обменных и аналитических форматах.

В отличие от работы, построенной вокруг отдельных конструкций языка Python, здесь синтаксис используется только как инструмент решения аналитической задачи. Каждая операция должна сопровождаться интерпретацией результата.

Студент отвечает не только на вопрос «что вернула функция?», но и на вопрос «что это означает для исследуемого набора данных?».

In [ ]:
print("Hello, World!")


## 1. Постановка задачи

### 1.1. Источник датасета

**Название файла:** `The number of crimes in Russia 2008-2023.csv`

**Источник:** [СДО РТУ МИРЭА](https://online-edu.mirea.ru/)

**Дата получения:** 08.09.2026

**Период данных:** 2008-2023 (16 лет)

**Охват:** Российская Федерация в целом

### 1.2. Цель анализа

**Главная цель:** Провести анализ данных о количестве зарегистрированных преступлений в России за период 2008-2023 годов для выявления ключевых тенденций и закономерностей.

**Конкретные задачи:**

1. **Анализ временных трендов** - проследить, как менялось общее число преступлений по годам (2008-2023).

2. **Сравнение типов преступлений** - выявить наиболее и наименее распространённые категории преступлений.

3. **Региональный анализ** - определить регионы с наибольшей и наименьшей преступностью в абсолютных числах и на 100 000 населения.

4. **Анализ структуры** - рассчитать долю каждого типа преступления в общем объёме.

5. **Проверка качества данных** - выявить пропуски, аномалии, особенности.

### 1.3. Краткая предметная постановка

**Предметная область:** Криминология и статистика правонарушений.

**Объект исследования:** Зарегистрированные преступления на территории Российской Федерации.

**Ключевые вопросы исследования:**

1. Снижается ли преступность в России в целом за 16 лет?
2. Какие виды преступлений доминируют?
3. Наблюдается ли региональное неравенство?
4. Как отличается динамика разных типов преступлений?

**Ограничения данных:**
- Отсутствуют данные по Крыму и Севастополю до 2014 г.
- Нет данных по экономическим преступлениям за 2008–2009 гг.
- Отсутствует информация о раскрываемости преступлений.
- Нет данных о демографической структуре регионов.

## 2. Паспорт данных

В данном разделе формируется паспорт набора данных - формальное описание его структуры, состава и характеристик. Паспорт данных позволяет зафиксировать исходное состояние набора, оценить его пригодность для анализа и определить направления дальнейшей предобработки.

In [ ]:
from pathlib import Path
import pandas as pd


DATA_PATH = Path("The number of crimes in Russia 2008-2023.csv")
df = pd.read_csv(DATA_PATH,
                 delimiter=",",
                 decimal=".",
                 na_values="-200")

df.describe(include="all").round(2).T


### 2.1. Размер

In [ ]:
print(df.shape)
print(df.columns.tolist())
print(f"{DATA_PATH.stat().st_size / 1024:.2f} KiB")
df.head()


| Параметр | Значение |
|----------|----------|
| Число строк | 12,384 |
| Число столбцов | 5 |
| Размер файла | 589.7 KB |

### 2.2. Признаки

**Всего признаков:** 5

- **Категориальные (2):** Type of crime, Region
- **Числовые (3):** Year, The number of crimes, The number of crimes (cases per 100.000 population)

### 2.3. dtypes - типы данных

In [ ]:
import io


buf = io.StringIO()
df.info(buf=buf, memory_usage="deep")

print(buf.getvalue())


| Признак | Тип | Количество |
|---------|-----|----------|
| Type of crime | object | 12384 |
| Region | object | 12384 |
| Year | int64 | 12384 |
| The number of crimes | float64 | 12108 |
| The number of crimes (cases per 100.000 population) | float64 | 12092 |

Объём памяти: 2 MB

### 2.4. Словарь полей

| Поле | Тип | Единица | Описание | Пример |
|------|-----|---------|----------|--------|
| Type of crime | object | - | Тип преступления (8 категорий) | Murder, Theft |
| Region | object | - | Субъект РФ (включая "Russian Federation" как агрегат) | Moscow, Tuva Republic |
| Year | int | год | Год наблюдения (2008-2023) | 2008, 2015, 2023 |
| The number of crimes | float | штук | Абсолютное число зарегистрированных преступлений | 19740, 1326342 |
| The number of crimes (cases per 100.000 population) | float | на 100 тыс. | Число преступлений на 100 000 населения | 13.8, 929.2 |

### 2.5. Выводы

Набор данных представляет собой структурированную панель вида (тип преступления × регион × год), содержащую 12 384 строк и 5 признаков за период 2008–2023 гг. Данные **пригодны для разведочного анализа**, а структура панели позволяет отвечать на ключевые исследовательские вопросы о временных трендах, региональных различиях и сравнении типов преступлений; детальная проверка качества и оптимизация будут выполнены в следующем разделе.

## 3. Качество и предобработка

В данном разделе проводится детальная проверка качества данных: анализ пропусков, дубликатов, корректности типов данных, а также обоснованное преобразование признаков для дальнейшего анализа.

### 3.1. Пропуски, дубликаты и уникальные значения

In [ ]:
missing = pd.DataFrame({
    "Пропусков": df.isna().sum(),
    "Доля (%)": (df.isna().sum() / len(df) * 100).round(2)
})
print(missing)

print(f"\nВсего строк: {len(df):,}")
print(f"Строк с пропусками: {df.isna().any(axis=1).sum():,}")

print("\nПолных дубликатов:", df.duplicated().sum())

print("\nЧисло уникальных значений:")
print(df.nunique())


В наборе данных обнаружено <u>292 пропусков</u> (2.36% строк), все - в двух числовых столбцах: `The number of crimes` и `The number of crimes (cases per 100.000 population)`. В категориальных и временном столбцах пропусков нет.

Пропуски решено НЕ заполнять и НЕ удалять, поскольку:

1. **Заполнение исказит данные** - например, если поставить `0`, то это будет означать «преступлений не было», что неверно; на самом деле данных просто нет.
2. **Удаление строк приведёт к потере информации** о других типах преступлений и годах.
3. **Пропуски имеют объективную историческую природу** - они отражают реальную ситуацию с данными, а не ошибку сбора.

Пропуски будут автоматически исключены при агрегации (например, `groupby().mean()` игнорирует NaN) и при расчёте корреляций.

Проверка показала <u>полное отсутствие дубликатов</u>, каждая комбинация уникальна и встречается в наборе ровно один раз. Такая структура соответствует «панельному» формату данных и не требует удаления каких-либо строк.

### 3.2. Типы данных, преобразования и их обоснование

| Столбец | Текущий dtype | Проблема |
|---------|---------------|----------|
| Type of crime | object | Избыточно по памяти |
| Region | object | Избыточно по памяти |
| Year | int64 | Можно на int16 |
| The number of crimes | float64 | оставлен как `float64`, поскольку содержит NaN |
| The number of crimes (cases per 100.000) | float64 | оставлен как `float64`, поскольку содержит дробные значения |

1. **`Type of crime` → `category`**:
   - Всего 9 уникальных значений, повторяются 12 384 раза
   - В `object` каждая строка хранится как Python-строка
   - В `category` хранятся только коды (int8) + таблица категорий

2. **`Region` → `category`**:
   - 86 уникальных значений, повторяются многократно
   - Аналогично `Type of crime`, экономия существенная

3. **`Year` → `int16`**:
   - Диапазон 2008-2023 помещается в int16 (от -32 768 до 32 767)
   - `int64` избыточен (8 байт вместо 2)

In [ ]:
before = df.memory_usage(deep=True).sum()

clean_df = df.copy()

# Преобразование категориальных признаков
clean_df["Type of crime"] = clean_df["Type of crime"].astype("category")
clean_df["Region"] = clean_df["Region"].astype("category")

# Оптимизация Year
clean_df["Year"] = clean_df["Year"].astype("int16")

after = clean_df.memory_usage(deep=True).sum()

print(f"До: {before / 1024**2:.2f} MiB")
print(f"После: {after / 1024**2:.2f} MiB")
print(f"Снижение: {(1 - after / before) * 100:.1f}%")


### 3.3. Выводы

Выполнена минимальная, но обоснованная предобработка: проверено 276 пропусков, проверены дубликаты, оптимизированы типы данных и получилась экономия 87.9% памяти, данные сохранены в новой переменной `clean_df`, исходный `df` не изменён.

## 4. EDA - Разведочный анализ